In [ ]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [ ]:
df = pd.read_csv(
    "household_power_consumption.txt",
    sep=";",    
    infer_datetime_format=True
)

In [ ]:
df.info()

In [ ]:
df.iloc[:, 2:-1] = df.iloc[:,2:-1].replace('?',np.nan)

In [ ]:
df.iloc[:, 2:-1].isnull().sum()

In [ ]:
df_clean = df

In [ ]:
cols = df_clean.columns[2:-1]
df_clean[cols] = df_clean[cols].apply(lambda x: pd.to_numeric(x.str.strip(), errors='coerce'))

In [ ]:
df_clean = df.dropna()

In [ ]:
df_clean[["Date","Time"]]

In [ ]:
df_clean["Day"] = df_clean["Date"].astype(str).str.split("/").str[0].astype(int)
df_clean["Month"] = df_clean["Date"].astype(str).str.split("/").str[1].astype(int)
df_clean["Hour"] = df_clean["Time"].astype(str).str.split(":").str[0].astype(int)

In [ ]:
df_clean.drop("Date",inplace=True,axis=1)
df_clean.drop("Time",inplace=True,axis=1)

In [ ]:
X = df_clean.drop("Global_active_power", axis=1)
y = df_clean["Global_active_power"]

In [ ]:
from sklearn.linear_model import LinearRegression
import xgboost as xg
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=44)

models = {
    "xgb": xg.XGBRegressor(),
    "LinearRegression": LinearRegression(),

}

results = {}

for name, model in models.items():
    pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", model)
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    results[name] = r2


In [ ]:
best_model = max(results,key=results.get)
best_model = models[best_model]

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("best_model", best_model)
    ])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
y_pred

In [ ]:
import joblib


In [ ]:
model = joblib.load("best_model.pkl")
y_pred = model.predict(X_test)
y_pred